# Query no Databricks (serverless) via Databricks Connect

**Kernel:** selecione o interpretador `connect/.venv` (canto superior direito > Select Kernel > Python Environments > `.venv` desta pasta `connect`).

Usa o profile `community` (workspace dbc-31b4ebc3-548d) e compute serverless. A primeira query pode levar alguns segundos por causa do cold start do serverless.

In [11]:
from databricks.connect import DatabricksSession
from databricks.sdk.core import Config

# auth_type="pat" forçado: evita depender do broker local de credenciais da
# extensão Databricks do VS Code (auth_type=metadata-service) -- se esse broker
# cair ou mudar de porta, a sessão quebra com ConnectionRefusedError mesmo com
# profile="community" configurado, porque env vars injetadas pela extensão têm
# prioridade sobre o profile. Forçar PAT aqui ignora essas env vars.
#
# .serverless(True) do builder conta como "connection parameter explícito" e
# não pode ser combinado com sdkConfig() -- por isso serverless_compute_id é
# setado direto no Config, sem chamar .serverless() no builder.
cfg = Config(profile="community", auth_type="pat")
cfg.serverless_compute_id = "auto"
spark = DatabricksSession.builder.sdkConfig(cfg).getOrCreate()
spark

In [ ]:
# Query de teste
df = spark.sql("SELECT 1 AS ok, current_catalog() AS catalog, current_user() AS user")
df.show()

In [ ]:
# Explorar o que existe no workspace
spark.sql("SHOW CATALOGS").show()
# spark.sql("SHOW SCHEMAS IN workspace").show()
# spark.sql("SELECT * FROM workspace.default.minha_tabela LIMIT 10").show()

In [ ]:
spark.sql(""" create catalog if not exists advworks_dev""")

In [ ]:
spark.sql(""" select date_sub('2025-06-15', -7)""").show()

In [ ]:
spark.conf.get("spark.sql.shuffle.partitions")

In [ ]:
df = spark.read.table("advworks.raw.customer")

df.write.mode("parquet").options